# Q3/Q4 analysis and mixed-precision follow-up
This notebook consolidates the verified sweep and prints the next reproducible commands. It does not claim integer-runtime speedup.

In [ ]:
import pandas as pd
from pathlib import Path
summary = pd.read_csv(Path('../results/tables/compression_summary.csv'))
display(summary)


In [ ]:
ax = summary.plot.scatter(x='artifact_mib', y='test_accuracy_percent', s=90, grid=True, title='Accuracy–storage Pareto sweep')
for _, r in summary.iterrows():
    ax.annotate(r.policy, (r.artifact_mib, r.test_accuracy_percent), xytext=(5,5), textcoords='offset points')
ax.set_xlabel('Packed artifact size (MiB)')
ax.set_ylabel('Held-out test top-1 (%)')


## Recommended experiments
Keep activations at A6; test selective W6 depthwise and W8 stem/classifier exceptions. Select on validation accuracy versus actual bytes, then evaluate test once.

In [ ]:
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'
commands = [
 f'python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir {DATA_DIR} --device cuda --weight-bits 4 --activation-bits 6 --first-last-weight-bits 8 --epochs 12 --run-name mp-w4a6-edgew8-seed6886',
 f'python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir {DATA_DIR} --device cuda --weight-bits 4 --activation-bits 6 --depthwise-weight-bits 6 --first-last-weight-bits 8 --epochs 12 --run-name mp-w4dw6edgew8-a6-seed6886',
 f'python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir {DATA_DIR} --device cuda --weight-bits 4 --activation-bits 6 --edge-bits 8 --depthwise-weight-bits 6 --first-last-weight-bits 8 --epochs 12 --run-name mp-w4dw6edgew8-a6edge8-seed6886'
]
print('\n\n'.join(commands))


## W&B Parallel Coordinates data
The next cell creates one completed W&B run per existing policy. Set `WANDB_PROJECT`, execute it once, then add a Parallel Coordinates panel in the W&B workspace using these logged columns. Do not substitute the static Pareto plot for this mandatory assignment item.

In [ ]:
# !pip install -q wandb
import os
import wandb
WANDB_PROJECT = os.environ.get('WANDB_PROJECT', 'cs6886-assignment2-quantization')
for row in summary.to_dict(orient='records'):
    run = wandb.init(project=WANDB_PROJECT, name=f"artifact-{row['policy'].lower()}", config={'source': 'verified_artifact_handoff', 'policy': row['policy']}, reinit=True)
    wandb.log(row)
    run.finish()
print('Logged', len(summary), 'artifact-summary runs to', WANDB_PROJECT)
